## Financial Fraud Project

## Project Goal
In this notebook I am cleaning and preparing the dataset so it 
is ready to be used for machine learning. This includes removing 
unnecessary columns, handling duplicates, encoding categorical 
variables, and engineering new features.

## 1.Data Cleaning & Pre-processing

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('Finanacial dataset project.csv')
print(f'Shape: {df.shape}')
df.head()

Shape: (6362620, 11)


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


## 2. Check for Nulls & Duplicates


Here I am checking if there are any missing values or 
duplicate rows that need to be cleaned up.

In [2]:
print('Null values:')
print(df.isnull().sum())
print()
print(f'Duplicate rows: {df.duplicated().sum()}')

# Drop duplicates if any
df = df.drop_duplicates()
print(f'Shape after dropping duplicates: {df.shape}')

Null values:
step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64

Duplicate rows: 0
Shape after dropping duplicates: (6362620, 11)


## 3. Drop Unnecessary Columns
I am removing these columns based on what I found in my EDA:
- nameOrig and nameDest are just account ID numbers that the 
model cannot learn anything useful from
- isFlaggedFraud barely catches any real fraud and could 
confuse the model

In [32]:
df = df.drop(columns=['nameOrig', 'nameDest', 'isFlaggedFraud'], errors='ignore')
print(f'Remaining columns: {df.columns.tolist()}')

Remaining columns: ['step', 'type', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'isFraud']


## 4. Encode Categorical Features
The type column contains text categories so I need to convert 
it into numbers using one-hot encoding so the model can read it.

In [33]:
# One-hot encode the 'type' column
df = pd.get_dummies(df, columns=['type'], drop_first=True)
print(f'Shape after encoding: {df.shape}')
df.head()

Shape after encoding: (6362620, 11)


,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER
0,1,9839.64,170136.0,160296.36,0.0,0.0,0,False,False,True,False
1,1,1864.28,21249.0,19384.72,0.0,0.0,0,False,False,True,False
2,1,181.00,181.0,0.00,0.0,0.0,1,False,False,False,True
3,1,181.00,181.0,0.00,21182.0,0.0,1,True,False,False,False
4,1,11668.14,41554.0,29885.86,0.0,0.0,0,False,False,True,False


## 5. Feature Engineering
Here I am creating new columns based on patterns I noticed 
in the EDA that might help the model detect fraud better.

In [34]:
# Was the origin account drained to zero?
df['orig_balance_zero'] = (df['newbalanceOrig'] == 0).astype(int)

# Balance discrepancy at origin account
df['balance_disc_orig'] = (df['oldbalanceOrg'] - df['amount']) - df['newbalanceOrig']

# Log-transformed amount
df['log_amount'] = np.log1p(df['amount'])

print(f'Final shape: {df.shape}')
df.head()

Final shape: (6362620, 14)


,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER,orig_balance_zero,balance_disc_orig,log_amount
0,1,9839.64,170136.0,160296.36,0.0,0.0,0,False,False,True,False,0,0.0,9.194276
1,1,1864.28,21249.0,19384.72,0.0,0.0,0,False,False,True,False,0,0.0,7.531166
2,1,181.00,181.0,0.00,0.0,0.0,1,False,False,False,True,1,0.0,5.204007
3,1,181.00,181.0,0.00,21182.0,0.0,1,True,False,False,False,1,0.0,5.204007
4,1,11668.14,41554.0,29885.86,0.0,0.0,0,False,False,True,False,0,0.0,9.364703


## 6. Save Pre-processed Dataset
Now that the data is clean I am saving it as a new CSV file 
so I can load it in Notebook 3 for modeling.

In [35]:
df.to_csv('fraud_preprocessed.csv', index=False)
print('Saved to fraud_preprocessed.csv')
print(f'Final shape: {df.shape}')

Saved to fraud_preprocessed.csv
Final shape: (6362620, 14)
